# GoTriple Stats - Facet Counts

Query the GoTriple API for aggregation counts on various facets:
- **resource-type** (type)
- **discipline** (topic)
- **source** (provider)
- **subjects** (TBD)

In [1]:
import requests
import json
from datetime import datetime

In [2]:
BASE_URL = "https://api.gotriple.eu/api/documents"

FACETS = [
    {"facet": "resource-type", "aggs": "type"},
    {"facet": "discipline",    "aggs": "topic"},
    {"facet": "source",        "aggs": "provider"},
]

In [3]:
def fetch_facet_counts(aggs_param):
    """Query GoTriple API and return list of {key, doc_count} buckets."""
    resp = requests.get(BASE_URL, params={"aggs": aggs_param}, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    aggs = data.get("aggs", {})
    facet_data = aggs.get(aggs_param, {})
    return facet_data.get("buckets", [])

In [4]:
results = []

for facet_def in FACETS:
    facet_name = facet_def["facet"]
    aggs_param = facet_def["aggs"]
    print(f"Querying facet: {facet_name} (aggs={aggs_param})...")
    
    buckets = fetch_facet_counts(aggs_param)
    
    if buckets:
        total = sum(b["doc_count"] for b in buckets)
        results.append({
            "facet": facet_name,
            "count": total,
            "buckets": [{"key": b["key"], "count": b["doc_count"]} for b in buckets]
        })
        print(f"  -> {len(buckets)} buckets, total count: {total}")
    else:
        results.append({"facet": facet_name, "count": 0, "buckets": []})
        print(f"  -> No data returned")

print("\nDone.")

Querying facet: resource-type (aggs=type)...
  -> 20 buckets, total count: 23366316
Querying facet: discipline (aggs=topic)...
  -> 27 buckets, total count: 40042322
Querying facet: source (aggs=provider)...
  -> 19 buckets, total count: 23283036

Done.


In [5]:
# Preview results
for r in results:
    print(f"\n=== {r['facet']} (total: {r['count']}) ===")
    for b in r["buckets"]:
        print(f"  {b['key']:40s} {b['count']:>12,}")


=== resource-type (total: 23366316) ===
  typ_article                                10,389,394
  typ_text                                    4,692,131
  typ_thesis                                  2,172,443
  other                                       2,154,968
  typ_conference                                751,856
  typ_book                                      727,600
  typ_report                                    644,187
  typ_image                                     384,934
  typ_blog-post                                 336,654
  typ_book-part                                 336,322
  typ_dataset                                   282,575
  typ_review                                    258,444
  undefined                                     134,712
  typ_manuscript                                 42,836
  typ_preprint                                   19,558
  typ_periodical                                 17,367
  typ_learning-object                            14,479
  typ_s

In [ ]:
# Save to JSON
output_file = "gotriple_facet_counts.json"

output = {
    "generated_at": datetime.now().isoformat(),
    "facets": [{"facet": r["facet"], "count": r["count"]} for r in results],
    "details": results
}

with open(output_file, "w") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Saved to {output_file}")